# MTG Deck Composition Predictor - Training Notebook

Train the Deck Composition Predictor to learn which cards belong in which deck archetypes.
Uses BPR ranking loss with leave-one-out archetype pooling for per-archetype card discrimination.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU` before running.

## 1. Setup

In [ ]:
# Mount Google Drive (project lives here)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys

PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install third-party dependencies only
!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Data Validation

In [ ]:
from src.config import GRAPH_PATH, METAGAME_PARQUET, DECKLISTS_PARQUET
import pandas as pd

for name, path in [('Metagame', METAGAME_PARQUET),
                    ('Decklists', DECKLISTS_PARQUET), ('Graph', GRAPH_PATH)]:
    if path.exists():
        if path.suffix == '.parquet':
            rows = len(pd.read_parquet(path))
            print(f'  OK  {name}: {rows:,} rows')
        else:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f'  OK  {name}: {size_mb:.1f} MB')
    else:
        print(f'  MISSING  {name} — run: python -m src.ingest_all')

assert GRAPH_PATH.exists(), 'Graph not found! Run data ingestion first.'

# Check board column
dl = pd.read_parquet(DECKLISTS_PARQUET)
if 'board' in dl.columns:
    main_count = (dl['board'] == 'main').sum()
    side_count = (dl['board'] == 'side').sum()
    print(f'\n  Board split: {main_count:,} mainboard / {side_count:,} sideboard entries')

# Show metagame share for eligible archetypes
meta = pd.read_parquet(METAGAME_PARQUET)
latest_share = meta.sort_values('snapshot_date').groupby('archetype')['meta_share_pct'].last().sort_values(ascending=False)
print(f'\nMetagame share (latest snapshot):')
for arch, share in latest_share.head(15).items():
    marker = ' ***' if share >= 5.0 else ''
    print(f'  {share:5.1f}%  {arch}{marker}')
print(f'\nTotal archetypes: {len(latest_share)} | With >= 5% meta share: {(latest_share >= 5.0).sum()}')

print('\nData ready for training.')

## 3. Hyperparameters

Tune these before training. Restart runtime and re-run from this cell to retrain with new values.

In [ ]:
# ── Hyperparameters (edit these to experiment) ──

# Model architecture
HIDDEN_DIM = 64          # Hidden dimension (default: 128, try 32-64 for less overfitting)
NUM_HEADS = 4            # HGT attention heads (default: 4)
NUM_HGT_LAYERS = 2      # HGT layers (default: 3, try 1-2 for less oversmoothing)
DROPOUT = 0.3            # Dropout rate (default: 0.3)

# BPR training
LEARNING_RATE = 0.001    # Peak learning rate (default: 0.001)
WEIGHT_DECAY = 0.001     # L2 regularization (default: 0.0001, try 0.001-0.01)
NUM_EPOCHS = 50          # Max training epochs (early stopping may end sooner)
N_NEGATIVES = 50         # Negatives per positive in BPR (default: 50)
BPR_MARGIN = 1.0         # BPR margin (default: 1.0)
PATIENCE = 10            # Early stopping patience (default: 10)
FREEZE_HGT = False       # Freeze HGT backbone (default: False)
RECENCY_DAYS = 30        # Use decklists from last N days (default: 30)
CHECKPOINT_METRIC = 'val_hits10'  # Metric to optimize (val_hits10, val_hits15, f1, mrr)

# Learning rate schedule
WARMUP_EPOCHS = 5        # Linear warmup from ~0 to LEARNING_RATE (default: 5, 0 to disable)
LR_SCHEDULER = 'cosine'  # Schedule after warmup: cosine | linear | none

print('Hyperparameters set.')
print(f'  Architecture: hidden_dim={HIDDEN_DIM}, heads={NUM_HEADS}, layers={NUM_HGT_LAYERS}, dropout={DROPOUT}')
print(f'  Training:     lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}, epochs={NUM_EPOCHS}')
print(f'  BPR:          n_neg={N_NEGATIVES}, patience={PATIENCE}, freeze_hgt={FREEZE_HGT}')
print(f'  LR Schedule:  {WARMUP_EPOCHS}-epoch warmup → {LR_SCHEDULER} decay')

## 4. Train Deck Predictor (GPU-Accelerated)

In [ ]:
import os, time
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
from src.train_deck import train_deck

start_time = time.time()
result = train_deck(
    device=device,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_hgt_layers=NUM_HGT_LAYERS,
    dropout=DROPOUT,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=NUM_EPOCHS,
    n_negatives=N_NEGATIVES,
    bpr_margin=BPR_MARGIN,
    patience=PATIENCE,
    freeze_hgt=FREEZE_HGT,
    recency_days=RECENCY_DAYS,
    checkpoint_metric=CHECKPOINT_METRIC,
    warmup_epochs=WARMUP_EPOCHS,
    lr_scheduler=LR_SCHEDULER,
)
elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed:.1f}s ({elapsed/NUM_EPOCHS:.2f}s/epoch)')

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

train_losses = result['train_losses']
val_metrics_history = result['val_metrics_history']
best_epoch = result['best_epoch']
n_epochs = result['hp']['num_epochs']

val_epochs = [1] + list(range(5, n_epochs + 1, 5))
val_hits10 = [m.get('hits_at_10', 0) for m in val_metrics_history]
val_hits15 = [m.get('hits_at_15', 0) for m in val_metrics_history]
val_mrr = [m.get('mrr', 0) for m in val_metrics_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# BPR Loss
ax1.plot(range(1, len(train_losses) + 1), train_losses, alpha=0.4, label='BPR Train Loss')
ax1.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BPR Loss'); ax1.set_title('Training Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Ranking Metrics
ax2.plot(val_epochs[:len(val_hits10)], val_hits10, 'o-', color='green', markersize=3, label='Hits@10')
ax2.plot(val_epochs[:len(val_hits15)], val_hits15, 's-', color='blue', markersize=3, label='Hits@15')
ax2.plot(val_epochs[:len(val_mrr)], val_mrr, '^:', color='purple', markersize=3, label='MRR')
ax2.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score'); ax2.set_title('Validation Ranking Metrics')
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Embedding Diagnostics

Check whether archetype embeddings are actually different from each other. If cosine similarities are all close to 1.0, the model can't distinguish archetypes.

In [ ]:
import torch.nn.functional as F
from src.train_deck import _get_eligible_archetypes

model = result['model']
data = result['data']
model.eval()

with torch.no_grad():
    output = model(data)
    node_emb = output['node_embeddings']

arch_emb = node_emb['archetype']
card_emb = node_emb['card']
arch_names = data['archetype'].names
card_names = data['card'].names

arch_to_idx = {a: i for i, a in enumerate(arch_names)}
eligible = sorted(_get_eligible_archetypes(arch_to_idx))

if len(eligible) < 2:
    print('Not enough eligible archetypes for comparison')
else:
    # Archetype embedding similarity
    elig_emb = arch_emb[eligible]
    elig_names = [arch_names[i] for i in eligible]
    cos_sim = F.cosine_similarity(elig_emb.unsqueeze(0), elig_emb.unsqueeze(1), dim=-1)

    print(f'=== Archetype Embedding Cosine Similarity ({len(eligible)} eligible) ===')
    print(f'  Mean pairwise similarity: {cos_sim.mean():.4f}')
    mask = ~torch.eye(len(eligible), dtype=torch.bool, device=cos_sim.device)
    off_diag = cos_sim[mask]
    print(f'  Mean (excl self):         {off_diag.mean():.4f}')
    print(f'  Min:                      {off_diag.min():.4f}')
    print(f'  Max:                      {off_diag.max():.4f}')
    print(f'  Std:                      {off_diag.std():.4f}')
    print()

    if off_diag.mean() > 0.95:
        print('  WARNING: Archetype embeddings are nearly identical!')
    elif off_diag.mean() > 0.8:
        print('  Moderate similarity — some differentiation but could be better.')
    else:
        print('  Good diversity — archetypes have distinct embeddings.')

    print(f'\nPairwise similarities:')
    header = f'{"":20s} ' + ' '.join(f'{n[:8]:>8s}' for n in elig_names)
    print(header)
    for i, name in enumerate(elig_names):
        row = f'{name:20s} ' + ' '.join(f'{cos_sim[i,j]:.4f}  ' for j in range(len(elig_names)))
        print(row)

    # Card embedding diversity
    sample_cards = torch.randperm(card_emb.shape[0])[:100]
    sample_emb = card_emb[sample_cards]
    card_cos = F.cosine_similarity(sample_emb.unsqueeze(0), sample_emb.unsqueeze(1), dim=-1)
    card_mask = ~torch.eye(100, dtype=torch.bool, device=card_cos.device)
    card_off = card_cos[card_mask]
    print(f'\n=== Card Embedding Diversity (100 random cards) ===')
    print(f'  Mean pairwise similarity: {card_off.mean():.4f}')
    print(f'  Std:                      {card_off.std():.4f}')

## 7. Evaluation & Dashboard

In [ ]:
from src.train_deck import evaluate_deck

eval_results = evaluate_deck(
    model=result['model'],
    data=result['data'],
    arch_pos_cards=result['arch_pos_cards'],
    arch_neg_pool=result['arch_neg_pool'],
    train_cards=result['train_cards'],
    val_cards=result['val_cards'],
    test_cards=result['test_cards'],
    run_dir=result['run_dir'],
    model_path=result['model_path'],
    train_losses=result['train_losses'],
    val_metrics_history=result['val_metrics_history'],
    best_epoch=result['best_epoch'],
    hp=result['hp'],
)

print(f"\nResults saved to: {result['run_dir']}")

In [ ]:
from src.visualize_deck import main as generate_deck_dashboard

generate_deck_dashboard(
    run_dir=result['run_dir'],
    model_path=result['model_path'],
)

print(f"\nAll results in: {result['run_dir']}")
!ls -la "{result['run_dir']}"